# Protocol lưu trữ / tham chiếu lịch sử
Notebook này giữ cấu hình cũ để truy xuất thí nghiệm. Lượt GĐ2 mới tập trung FedAvg dùng **kaggle_stage2_fedavg_v4.ipynb** với split content-aware v4. Không trộn trực tiếp accuracy của hai protocol để tính chênh lệch GĐ1–GĐ2.


# GĐ2 research matrix: 13 conditions, 3 training seeds, 117 jobs
MobileNetV3-Small, 38 classes; Centralized / FedAvg / Local-only. Training seeds: 42, 123, 2026. Split seed: 42. Required label alpha: 100, 10, 1, 0.5, 0.1; retain 5 and 0.05 as exploratory points. All declared quantity/feature/mixed conditions are included.
This is an expanded protocol, not evidence of completed training. The unchanged 30-hour budget guard may refuse this matrix after calibration. Never remove jobs or seeds silently to make it fit.
One seed contributes one Local-only client mean. summary.csv reports training-seed mean/std and 95% Student-t CI when n>=3. Stage-3 candidate uses validation gaps and remains blocked until the complete matrix is available.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

# EDIT THESE VALUES. Do not select a similar-looking dataset automatically.
PACKAGE_ROOT = Path('/kaggle/working/gd2_federated_learning')
DATASET_ROOT = Path('/kaggle/input/plantvillage-dataset/color')
PARTITION_INDEX = PACKAGE_ROOT / 'data/partitions_train_v3_research/index.json'
OUTPUT_ROOT = Path('/kaggle/working/stage2_research_output')
GPU_ALREADY_USED_HOURS = 0.0       # include earlier sessions/work outside this runner
KAGGLE_QUOTA_REMAINING_HOURS = None # enter the quota shown by Kaggle, or leave None
RESTORE_SOURCE = None  # e.g. Path('/kaggle/input/my-stage2-backup/stage2_research_output')
RECOVER_STALE_LOCK = False
INSTALL_TESTED_RUNTIME = False  # enable only if the dependency check below would fail
assert PACKAGE_ROOT.is_dir(), PACKAGE_ROOT
os.chdir(PACKAGE_ROOT)
sys.path.insert(0, str(PACKAGE_ROOT))

## Restore toàn bộ output trước khi chạy lại
`/kaggle/working` không bền qua session. Gắn Kaggle Dataset/output backup của phiên trước, đặt `RESTORE_SOURCE`, rồi chạy cell này. Phải restore nguyên cây gồm checkpoint, `budget_state.json`, config, log và comparison.

In [ ]:
if RESTORE_SOURCE is not None:
    RESTORE_SOURCE = Path(RESTORE_SOURCE)
    assert (RESTORE_SOURCE / 'budget_state.json').is_file(), 'Incomplete backup'
    if OUTPUT_ROOT.exists():
        assert (OUTPUT_ROOT / 'budget_state.json').is_file(), 'Refusing to merge into unrelated output'
        print('Output already restored; ledger was not overwritten.')
    else:
        shutil.copytree(RESTORE_SOURCE, OUTPUT_ROOT)
        print('Restored', RESTORE_SOURCE, 'to', OUTPUT_ROOT)

## GPU, dependency, dataset, manifest và leaf-map preflight

In [ ]:
if INSTALL_TESTED_RUNTIME:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch==2.6.0', 'torchvision==0.21.0',
                    '--index-url', 'https://download.pytorch.org/whl/cu124'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-kaggle.txt'], check=True)
import torch, torchvision, flwr, ray
print({'python': sys.version, 'torch': torch.__version__, 'torchvision': torchvision.__version__,
       'flwr': flwr.__version__, 'ray': ray.__version__, 'cuda': torch.cuda.is_available()})
subprocess.run(['nvidia-smi'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=True)
assert torch.__version__.split('+')[0].startswith('2.6.'), 'Use tested torch 2.6.x runtime'
assert torchvision.__version__.split('+')[0].startswith('0.21.'), 'Use tested torchvision 0.21.x pair'
assert flwr.__version__ == '1.36.0' and ray.__version__ == '2.55.1'
assert torch.cuda.is_available(), 'Enable one Kaggle GPU before calibration'
assert DATASET_ROOT.is_dir(), f'Exact dataset root missing: {DATASET_ROOT}'
assert PARTITION_INDEX.is_file(), f'Partition index missing: {PARTITION_INDEX}'
index = json.loads(PARTITION_INDEX.read_text(encoding='utf-8'))
assert index['total_source_images'] == 54305 and len(index['partitions']) == 6
class_dirs = sorted(p.name for p in DATASET_ROOT.iterdir() if p.is_dir())
image_count = sum(1 for p in DATASET_ROOT.rglob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.tif','.tiff'})
assert image_count == index['total_source_images'], (image_count, index['total_source_images'])
first_partition = PACKAGE_ROOT / next(iter(index['partitions'].values()))['relative_dir']
meta = json.loads((first_partition / 'partition_config.json').read_text(encoding='utf-8'))
assert class_dirs == sorted(meta['class_names']), 'Dataset class mapping differs; do not relabel'
leaf_audit = meta['leaf_group_audit']
assert leaf_audit['total_images'] == image_count
print({'images': image_count, 'classes': len(class_dirs), 'leaf_map_coverage': leaf_audit['leaf_map_coverage'],
       'test_hash': index['test_paths_sha256'], 'val_hash': index['val_paths_sha256']})

In [ ]:
COMMON = [sys.executable, '-m', 'fl_training.kaggle_budget', '--config', 'configs/stage2_research.yaml',
          '--dataset-root', str(DATASET_ROOT), '--partition-index', str(PARTITION_INDEX),
          '--output-root', str(OUTPUT_ROOT)]
USAGE = ['--already-used-hours', str(GPU_ALREADY_USED_HOURS)]
if KAGGLE_QUOTA_REMAINING_HOURS is not None:
    USAGE += ['--quota-remaining-hours', str(KAGGLE_QUOTA_REMAINING_HOURS)]
RECOVER = ['--recover'] if RECOVER_STALE_LOCK else []
subprocess.run(COMMON + ['--action', 'plan'], check=True)
plan = json.loads((OUTPUT_ROOT / 'requested_plan.json').read_text())
assert len(plan['jobs']) == 117
print('Dry-run validated', len(plan['jobs']), 'jobs')

## Calibration thật và khóa protocol
Đo cả ba phương pháp trên dữ liệu thật; Local-only chạy đủ 10 client. Test không được dùng để chọn round. Nếu dự toán không chứa được ít nhất 10 round cho cả ma trận, lệnh dừng trước main training. Hệ số dự phòng là 1.5.

In [ ]:
subprocess.run(COMMON + ['--action', 'calibrate'] + USAGE + RECOVER, check=True)
frozen = json.loads((OUTPUT_ROOT / 'frozen_plan.json').read_text())
assert 10 <= frozen['rounds'] <= 20
print(frozen)

## Run/resume ma trận
Mỗi lần chạy giới hạn chủ động 8 giờ và dừng nhận việc mới trước 15 phút. Exit code 75 nghĩa là đã pause theo ngân sách/session; lưu toàn bộ output rồi resume ở phiên sau. Không đổi round riêng cho job sau.

In [ ]:
result = subprocess.run(COMMON + ['--action', 'run'] + USAGE + RECOVER)
assert result.returncode in (0, 75), result.returncode
print('completed' if result.returncode == 0 else 'paused; persist OUTPUT_ROOT and resume later')

## Collect CPU-only và đóng gói output

In [ ]:
subprocess.run(COMMON + ['--action', 'collect'], check=True)
status = json.loads((OUTPUT_ROOT / 'comparison/collection_status.json').read_text())
print(status)
archive = shutil.make_archive('/kaggle/working/stage2_research_output', 'zip',
                              root_dir=OUTPUT_ROOT.parent, base_dir=OUTPUT_ROOT.name)
print('Download this ZIP or save it as a Kaggle Dataset before ending the session:', archive)